# Milestone 4 — Feature Engineering
## AI Bill Anomaly Detection System

EDA (Milestone 3) showed that **raw columns correlate weakly** with anomalies — invoices
aren't anomalous because one value crosses a fixed threshold, they're anomalous **relative
to context** (their department, their vendor, their category). This notebook builds the
features Isolation Forest actually needs.

**Features built here:**
1. `Invoice_Month`, `Invoice_Weekday` — temporal patterns
2. `Vendor_Frequency` — invoice count per vendor (catches shell-vendor-like behavior)
3. `Dept_Avg_Amount` — department's average invoice amount (the "normal" baseline)
4. `Amount_Deviation` — invoice amount ÷ department average (the key relative-anomaly signal)
5. `Amount_Category` — binned amount (Low/Medium/High/Very High)
6. `GST_Category` — whether GST falls in a standard slab (5/12/18/28%) or not
7. `Category_Avg_Amount` / `Category_Amount_Deviation` — same idea, but by item category

We also **encode categorical columns** so the feature set is fully numeric and ready for
Isolation Forest in Milestone 5.

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/processed/invoices_cleaned.csv", parse_dates=["Invoice_Date"])
print("Shape:", df.shape)
df.head()

Shape: (10200, 10)


,Invoice_ID,Vendor_Name,Department,Invoice_Date,Amount,GST,Quantity,Category,True_Anomaly,IQR_Outlier_Flag
0,INV200078,Vendor_030,Health,2024-01-13,10359.12,12.0,16,Vehicle Maintenance,1,0
1,INV103742,Vendor_031,Administration,2025-04-29,17481.48,5.0,5,Vehicle Maintenance,0,0
2,INV200068,Vendor_024,Water Supply,2025-04-17,21126.69,5.0,23,Electronics,1,0
3,INV105723,Vendor_039,Transport,2025-07-15,11991.82,5.0,33,Office Equipment,0,0
4,INV108987,Vendor_037,Transport,2024-07-12,13688.73,12.0,9,Construction Material,0,0


## 1. Temporal features

In [2]:
df["Invoice_Month"] = df["Invoice_Date"].dt.month
df["Invoice_Weekday"] = df["Invoice_Date"].dt.weekday  # 0 = Monday

df[["Invoice_Date", "Invoice_Month", "Invoice_Weekday"]].sample(5)

,Invoice_Date,Invoice_Month,Invoice_Weekday
7414,2025-11-18,11,1
8804,2024-03-02,3,5
1888,2024-05-25,5,5
6557,2024-03-05,3,1
7413,2025-01-28,1,1


## 2. Vendor frequency — catches shell-vendor-like behavior

In [3]:
df["Vendor_Frequency"] = df.groupby("Vendor_Name")["Invoice_ID"].transform("count")

print("Top 5 vendors by frequency:")
print(df.groupby("Vendor_Name")["Vendor_Frequency"].first().sort_values(ascending=False).head())

Top 5 vendors by frequency:
Vendor_Name
Vendor_030    296
Vendor_002    293
Vendor_026    279
Vendor_036    273
Vendor_037    269
Name: Vendor_Frequency, dtype: int64


## 3. Department-relative amount — the core anomaly signal

In [4]:
df["Dept_Avg_Amount"] = df.groupby("Department")["Amount"].transform("mean")
df["Amount_Deviation"] = df["Amount"] / df["Dept_Avg_Amount"]

# Sanity check: known injected high-amount anomalies should show high deviation
print("Amount_Deviation stats for True_Anomaly == 1 vs == 0:")
print(df.groupby("True_Anomaly")["Amount_Deviation"].describe()[["mean", "50%", "max"]])

Amount_Deviation stats for True_Anomaly == 1 vs == 0:
                  mean       50%       max
True_Anomaly                              
0             0.813506  0.811483  1.596814
1             3.072695  0.922652  9.956348


**Confirms the hypothesis from EDA:** anomalous invoices have a much higher mean and max
`Amount_Deviation` than normal ones — this single engineered feature should carry a lot of
the model's discriminating power.

## 4. Category-relative amount (same idea, different grouping)

In [5]:
df["Category_Avg_Amount"] = df.groupby("Category")["Amount"].transform("mean")
df["Category_Amount_Deviation"] = df["Amount"] / df["Category_Avg_Amount"]

df[["Category", "Amount", "Category_Avg_Amount", "Category_Amount_Deviation"]].sample(5)

,Category,Amount,Category_Avg_Amount,Category_Amount_Deviation
9916,Stationery,19815.74,18027.235835,1.099211
8444,Medical Supplies,11096.99,19628.029920,0.565364
6336,Fuel,12871.96,20822.788215,0.618167
311,Construction Material,20510.21,18624.887285,1.101226
2977,Cleaning Supplies,12821.32,20238.442452,0.633513


## 5. Amount category (binned)

In [6]:
def amount_bucket(x):
    if x < 5000:
        return "Low"
    elif x < 20000:
        return "Medium"
    elif x < 50000:
        return "High"
    else:
        return "Very High"

df["Amount_Category"] = df["Amount"].apply(amount_bucket)
df["Amount_Category"].value_counts()

Amount_Category
Medium       6916
High         2922
Very High     300
Low            62
Name: count, dtype: int64

## 6. GST category — standard slab vs. non-standard

In [7]:
STANDARD_GST_SLABS = {5, 12, 18, 28}

df["GST_Category"] = df["GST"].apply(lambda x: "Standard" if x in STANDARD_GST_SLABS else "Non-Standard")

print(df["GST_Category"].value_counts())
print("\nOverlap with True_Anomaly:")
print(pd.crosstab(df["GST_Category"], df["True_Anomaly"]))

GST_Category
Standard        10001
Non-Standard      199
Name: count, dtype: int64

Overlap with True_Anomaly:
True_Anomaly     0    1
GST_Category           
Non-Standard     0  199
Standard      9358  643


**Confirms:** almost all `Non-Standard` GST rows are true anomalies — this is a strong,
almost deterministic signal. Good to have alongside the softer `Amount_Deviation` signal,
since Isolation Forest benefits from a mix of strong and moderate signals rather than
relying on just one.

## 7. Encode categorical features for modeling

In [8]:
# Isolation Forest needs numeric input. We use simple label/one-hot encoding here —
# no ordinal meaning is assumed for Department/Category/GST_Category/Amount_Category.
categorical_cols = ["Department", "Category", "GST_Category", "Amount_Category"]

df_encoded = pd.get_dummies(df, columns=categorical_cols, prefix=categorical_cols)

# Keep the original readable columns too (not just the encoded dummies). We need
# Department/Category/GST_Category/Amount_Category as plain text for filtering and
# display in the Streamlit dashboard (Milestone 7) -- only the *dummy* versions are
# meant for model input, the originals are for humans reading the data.
for col in categorical_cols:
    df_encoded[col] = df[col].values

print("Shape before encoding:", df.shape)
print("Shape after encoding:", df_encoded.shape)
df_encoded.columns.tolist()

Shape before encoding: (10200, 19)
Shape after encoding: (10200, 43)


['Invoice_ID',
 'Vendor_Name',
 'Invoice_Date',
 'Amount',
 'GST',
 'Quantity',
 'True_Anomaly',
 'IQR_Outlier_Flag',
 'Invoice_Month',
 'Invoice_Weekday',
 'Vendor_Frequency',
 'Dept_Avg_Amount',
 'Amount_Deviation',
 'Category_Avg_Amount',
 'Category_Amount_Deviation',
 'Department_Administration',
 'Department_Education',
 'Department_Health',
 'Department_It & Electronics',
 'Department_Public Works',
 'Department_Sanitation',
 'Department_Transport',
 'Department_Water Supply',
 'Category_Cleaning Supplies',
 'Category_Construction Material',
 'Category_Electronics',
 'Category_Fuel',
 'Category_Furniture',
 'Category_Medical Supplies',
 'Category_Office Equipment',
 'Category_Software License',
 'Category_Stationery',
 'Category_Vehicle Maintenance',
 'GST_Category_Non-Standard',
 'GST_Category_Standard',
 'Amount_Category_High',
 'Amount_Category_Low',
 'Amount_Category_Medium',
 'Amount_Category_Very High',
 'Department',
 'Category',
 'GST_Category',
 'Amount_Category']

## 8. Final feature set review

In [9]:
# Columns we will NOT feed into the model directly:
# - Invoice_ID, Vendor_Name (identifiers, not predictive features)
# - Invoice_Date (already decomposed into Month/Weekday)
# - True_Anomaly (ground truth — reserved for evaluation in Milestone 6, not training)
# - IQR_Outlier_Flag (Milestone-2 diagnostic, not a model input)

exclude_cols = ["Invoice_ID", "Vendor_Name", "Invoice_Date", "True_Anomaly",
                "IQR_Outlier_Flag"] + categorical_cols  # readable text columns are for
                # display/filtering only; their one-hot dummy versions are the model inputs
model_feature_cols = [c for c in df_encoded.columns if c not in exclude_cols]

print(f"Total engineered/encoded feature columns for modeling: {len(model_feature_cols)}")
print(model_feature_cols)

Total engineered/encoded feature columns for modeling: 34
['Amount', 'GST', 'Quantity', 'Invoice_Month', 'Invoice_Weekday', 'Vendor_Frequency', 'Dept_Avg_Amount', 'Amount_Deviation', 'Category_Avg_Amount', 'Category_Amount_Deviation', 'Department_Administration', 'Department_Education', 'Department_Health', 'Department_It & Electronics', 'Department_Public Works', 'Department_Sanitation', 'Department_Transport', 'Department_Water Supply', 'Category_Cleaning Supplies', 'Category_Construction Material', 'Category_Electronics', 'Category_Fuel', 'Category_Furniture', 'Category_Medical Supplies', 'Category_Office Equipment', 'Category_Software License', 'Category_Stationery', 'Category_Vehicle Maintenance', 'GST_Category_Non-Standard', 'GST_Category_Standard', 'Amount_Category_High', 'Amount_Category_Low', 'Amount_Category_Medium', 'Amount_Category_Very High']


In [10]:
print("Missing values in final feature set:")
print(df_encoded[model_feature_cols].isnull().sum().sum(), "total nulls")

print("\nAll numeric dtypes:", df_encoded[model_feature_cols].dtypes.apply(lambda x: x.kind).unique())

Missing values in final feature set:
0 total nulls

All numeric dtypes: <StringArray>
['f', 'i', 'b']
Length: 3, dtype: str


## 9. Save engineered dataset

In [11]:
import os
os.makedirs("../data/processed", exist_ok=True)

# Save full version (with ground truth + IDs) for evaluation later
output_path = "../data/processed/invoices_features.csv"
df_encoded.to_csv(output_path, index=False)
print(f"Saved engineered feature set to {output_path}")
print(f"Shape: {df_encoded.shape}")

# Also save the list of model-ready feature columns, so Milestone 5 doesn't need to
# re-derive this list
import json
with open("../data/processed/model_feature_columns.json", "w") as f:
    json.dump(model_feature_cols, f, indent=2)
print("Saved model_feature_columns.json")

Saved engineered feature set to ../data/processed/invoices_features.csv
Shape: (10200, 43)
Saved model_feature_columns.json


## Summary

| Feature | Type | Purpose |
|---|---|---|
| `Invoice_Month`, `Invoice_Weekday` | Temporal | Captures seasonal/weekly patterns |
| `Vendor_Frequency` | Behavioral | Flags shell-vendor-like invoice bursts |
| `Amount_Deviation` | Relative | Core signal — amount vs. department norm |
| `Category_Amount_Deviation` | Relative | Same idea, by item category |
| `Amount_Category` (encoded) | Binned | Coarse amount tier |
| `GST_Category` (encoded) | Rule-derived | Near-deterministic anomaly flag |
| `Department`, `Category` (encoded) | Categorical | Context for the model |

`Invoice_ID`, `Vendor_Name`, `Invoice_Date`, `True_Anomaly`, and `IQR_Outlier_Flag` are
kept in the saved CSV for traceability and evaluation, but excluded from the model's
training features (`model_feature_columns.json` is the authoritative list).

## Notes for Milestone 5 (Isolation Forest)

- Load `data/processed/invoices_features.csv`
- Load `model_feature_columns.json` to select `X` for training
- Remember: **do not pass `True_Anomaly` into the model** — it's unsupervised, this
  column is only for evaluation in Milestone 6

Next notebook: `05_isolation_forest.ipynb`